In [1]:
import os
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import datetime
from pathlib import Path
import subprocess

In [2]:
# CELL 2 – Temporary workaround for r5py/Python 3.12 compatibility issue with Java/JPype
os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@21/21.0.11'

# Patch jpype before importing to handle bytes return from getDefaultJVMPath
import jpype
original_getDefaultJVMPath = jpype.getDefaultJVMPath

def patched_getDefaultJVMPath(*args, **kwargs):
    result = original_getDefaultJVMPath(*args, **kwargs)
    if isinstance(result, bytes):
        result = result.decode('utf-8')
    return result

jpype.getDefaultJVMPath = patched_getDefaultJVMPath

# Import r5py after patching jpype
import r5py

In [3]:
# CELL 3 – Load data

OSM_FILE = "../../data/geo/Toronto.osm.pbf"
GTFS_FILES = [
    "../../data/geo/TTC Routes and Schedules Data.zip",
    "../../data/geo/GO-GTFS.zip",
    "../../data/geo/UP-GTFS.zip"
]
ORIGINS_FILE = "../../src/data/venues-centroids.geo.json"

In [4]:
# CELL 4 - Set parameters

# Grid cell size (meters)
cell_size = 150

# Maximum travel time (minutes)
max_travel_time = 60

# Departure time
departure_time = datetime.datetime(
    2026, 6, 9, 
    14, 0, 0
    )

In [5]:
# CELL 5 – Create grid for commute time matrix calculation

# UTM 17N extent (meters)
minx = 604706
miny = 4819831
maxx = 657265
maxy = 4867556

# Create grid polygons
grid_cells = []

for x in np.arange(minx, maxx, cell_size):
    for y in np.arange(miny, maxy, cell_size):
        grid_cells.append(
            box(x, y, x + cell_size, y + cell_size)
        )

grid = gpd.GeoDataFrame(
    {"id": range(len(grid_cells))},
    geometry=grid_cells,
    crs="EPSG:26917"
)

# Remove cells that intersect with Lake Ontario (80%)
LAKE_ONTARIO = gpd.read_file("../../data/geo/lake-ontario.gpkg")
grid["intersection_area"] = grid.geometry.intersection(LAKE_ONTARIO.geometry[0]).area
grid = grid[grid["intersection_area"] / grid.geometry.area < 0.8].drop(columns="intersection_area")

grid = grid.to_crs("EPSG:4326")

# Create destinations GeoDataFrame with centroids
destinations = grid.copy()
destinations["geometry"] = destinations.geometry.centroid

/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_30504/1515565343.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  destinations["geometry"] = destinations.geometry.centroid


In [6]:
# CELL 6 – Load origins
origins = gpd.read_file(ORIGINS_FILE)
origins["id"] = origins["id"].astype(str)
origins = origins.to_crs("EPSG:4326")

In [7]:
# CELL 7 – Build transport network
transport_network = r5py.TransportNetwork(
    OSM_FILE,
    GTFS_FILES
)
transport_network

In [8]:
# CELL 8 – Compute travel time matrix for each origin and merge as columns onto the grid
grid_time = grid.copy()

for _, origin in origins.iterrows():
    origin_id = origin["id"]
    origin_gdf = gpd.GeoDataFrame(
        [{"id": origin_id}],
        geometry=[origin.geometry],
        crs="EPSG:4326"
    )

    travel_time_matrix = r5py.TravelTimeMatrix(
        transport_network,
        origins=origin_gdf,
        destinations=destinations,
        departure=departure_time,
        transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
    )

    grid_time = grid_time.merge(
        travel_time_matrix[["to_id", "travel_time"]].rename(
            columns={"travel_time": f"travel_time_{origin_id}"}
        ),
        left_on="id",
        right_on="to_id",
        how="left"
    ).drop(columns="to_id")

    if max_travel_time is not None:
        grid_time[f"travel_time_{origin_id}"] = grid_time[f"travel_time_{origin_id}"].where(
            grid_time[f"travel_time_{origin_id}"] <= max_travel_time
        )

    print(f"Done: {origin_id}")

Done: 1
Done: 2
Done: 3
Done: 4
Done: 5
Done: 6
Done: 7
Done: 8
Done: 9
Done: 10
Done: 11
Done: 12
Done: 13
Done: 14
Done: 15
Done: 16
Done: 17
Done: 18
Done: 19
Done: 20
Done: 21
Done: 22
Done: 23
Done: 24
Done: 25
Done: 26
Done: 27
Done: 28
Done: 29
Done: 30
Done: 31
Done: 32
Done: 33
Done: 34
Done: 35
Done: 36
Done: 37
Done: 38
Done: 39
Done: 40
Done: 41
Done: 42
Done: 43
Done: 44
Done: 45
Done: 46
Done: 47
Done: 48
Done: 49
Done: 50
Done: 51
Done: 52
Done: 53
Done: 54
Done: 55
Done: 56
Done: 57
Done: 58
Done: 59
Done: 60
Done: 61
Done: 62
Done: 63
Done: 64
Done: 65
Done: 66
Done: 67
Done: 68
Done: 69
Done: 70
Done: 71
Done: 72
Done: 73
Done: 74
Done: 75
Done: 76
Done: 77
Done: 78
Done: 79
Done: 80
Done: 81
Done: 82
Done: 83


In [9]:
# CELL 9 – Export grid as a tileset

OUTPUT_FILE = "../../data/geo/commute_time.geojson"
grid_time.to_file(OUTPUT_FILE, driver="geojson")

PMTILES_FILE = "../../static/commute_time.pmtiles"

subprocess.run([
    "tippecanoe",
    "-z17",
    "-o", PMTILES_FILE,
    "--extend-zooms-if-still-dropping",
    "--drop-densest-as-needed",
    "--no-feature-limit",
    "--no-tile-size-limit",
    OUTPUT_FILE,
    "--force"
], check=True)

Path(OUTPUT_FILE).unlink()

For layer 0, using name "commute_time"
77973 features, 24434237 bytes of geometry and attributes, 539912 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  17/36698/47767  
